In [ ]:
# ============================================
# CATEGORY-LEVEL DEMAND DATASET CREATION
# ============================================

print("\n" + "="*60)
print("CREATING CATEGORY-LEVEL DEMAND DATASET")
print("="*60)

# ----------------------------------------
# Aggregate to category-month level
# ----------------------------------------
print("\n📊 Aggregating to category-month level...")

category_demand_monthly = orders_full_extended.groupby(
    ['product_category_name_english', 'year_month', 'bucket']
).agg({
    'order_id': 'count',  # Quantity
    'price': 'mean',       # Average price
    'freight_value': 'mean',
    'review_score': 'mean',
    'product_id': 'nunique',  # Product diversity
    'seller_id': 'nunique',   # Seller competition
    'customer_unique_id': 'nunique'  # Customer reach
}).reset_index()

# Rename for clarity
category_demand_monthly.rename(columns={'order_id': 'quantity'}, inplace=True)

print(f"  • Total observations: {len(category_demand_monthly):,}")
print(f"  • Unique categories: {category_demand_monthly['product_category_name_english'].nunique()}")
print(f"  • Time periods: {category_demand_monthly['year_month'].nunique()}")
print(f"  • Buckets represented: {category_demand_monthly['bucket'].nunique()}")

# ----------------------------------------
# Create log-transformed variables
# ----------------------------------------
category_demand_monthly['log_quantity'] = np.log(category_demand_monthly['quantity'])
category_demand_monthly['log_price'] = np.log(category_demand_monthly['price'])

# Create time index
category_demand_monthly['month_index'] = pd.Categorical(category_demand_monthly['year_month']).codes

# ----------------------------------------
# Filter to categories with sufficient data
# ----------------------------------------
print("\n📋 Filtering to categories with sufficient observations...")

# Count observations per category
category_counts = category_demand_monthly.groupby('product_category_name_english').size()

# Keep categories with at least 12 months of data
sufficient_categories = category_counts[category_counts >= 12].index

category_regression_data = category_demand_monthly[
    category_demand_monthly['product_category_name_english'].isin(sufficient_categories)
].copy()

print(f"  • Categories with 12+ observations: {len(sufficient_categories)}")
print(f"  • Total observations for regression: {len(category_regression_data):,}")

# ----------------------------------------
# Summary by bucket
# ----------------------------------------
print("\n📊 Category distribution by bucket:")
bucket_summary = category_regression_data.groupby('bucket').agg({
    'product_category_name_english': 'nunique',
    'quantity': ['count', 'mean'],
    'price': 'mean'
}).round(2)
print(bucket_summary)

# ----------------------------------------
# Focus on top 3 buckets
# ----------------------------------------
focus_buckets = ['LEISURE_LIFESTYLE', 'HOME_ESSENTIALS', 'PERSONAL_CARE']

category_focus = category_regression_data[
    category_regression_data['bucket'].isin(focus_buckets)
].copy()

print(f"\n🎯 Focus dataset (top 3 buckets):")
print(f"  • Observations: {len(category_focus):,}")
print(f"  • Categories: {category_focus['product_category_name_english'].nunique()}")
print(f"  • Avg obs per category: {len(category_focus) / category_focus['product_category_name_english'].nunique():.1f}")

# ----------------------------------------
# Show top categories by volume
# ----------------------------------------
print("\n📈 Top 15 categories by total quantity:")
top_categories = category_focus.groupby('product_category_name_english').agg({
    'quantity': 'sum',
    'bucket': 'first'
}).sort_values('quantity', ascending=False).head(15)

print(top_categories.to_string())

# ----------------------------------------
# Save dataset
# ----------------------------------------
category_focus.to_pickle('../data/processed/category_demand_monthly_focus.pkl')
print("\n✅ Category demand dataset created and saved!")
print(f"   Saved to: ../data/processed/category_demand_monthly_focus.pkl")

In [ ]:
# ============================================
# CATEGORY-LEVEL ELASTICITY ESTIMATION
# ============================================

import statsmodels.formula.api as smf

print("\n" + "="*60)
print("CATEGORY-LEVEL PRICE ELASTICITY ESTIMATION")
print("="*60)

# ----------------------------------------
# Pooled estimation (all categories)
# ----------------------------------------
print("\n" + "-"*60)
print("POOLED MODEL: All Categories with Bucket + Time FE")
print("-"*60)

pooled_model = smf.ols(
    'log_quantity ~ log_price + C(bucket) + C(month_index)',
    data=category_focus
).fit()

print(f"\n📊 Pooled Elasticity:")
print(f"  • Coefficient: {pooled_model.params['log_price']:.3f}")
print(f"  • Std Error: {pooled_model.bse['log_price']:.3f}")
print(f"  • t-statistic: {pooled_model.tvalues['log_price']:.3f}")
print(f"  • p-value: {pooled_model.pvalues['log_price']:.4f}")
print(f"  • R-squared: {pooled_model.rsquared:.3f}")
print(f"  • Observations: {pooled_model.nobs:.0f}")

# 95% CI
conf_int = pooled_model.conf_int(alpha=0.05)
ci_lower = conf_int.loc['log_price', 0]
ci_upper = conf_int.loc['log_price', 1]
print(f"  • 95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")

if pooled_model.pvalues['log_price'] < 0.05:
    print(f"\n✅ Statistically significant at 5% level!")
else:
    print(f"\n⚠️  Not statistically significant")

# ----------------------------------------
# Category-specific elasticities
# ----------------------------------------
print("\n" + "="*60)
print("CATEGORY-SPECIFIC ELASTICITIES")
print("="*60)

category_results = []

# Get top 20 categories by observation count
top_20_categories = category_focus.groupby('product_category_name_english').size().sort_values(ascending=False).head(20).index

for category in top_20_categories:
    cat_data = category_focus[category_focus['product_category_name_english'] == category]
    
    # Need at least 15 observations for reliable estimation
    if len(cat_data) < 15:
        continue
    
    try:
        # Estimate with time FE (bucket is constant within category)
        model = smf.ols('log_quantity ~ log_price + C(month_index)', data=cat_data).fit()
        
        elasticity = model.params['log_price']
        std_err = model.bse['log_price']
        t_stat = model.tvalues['log_price']
        p_value = model.pvalues['log_price']
        
        # 95% CI
        conf_int = model.conf_int(alpha=0.05)
        ci_lower = conf_int.loc['log_price', 0]
        ci_upper = conf_int.loc['log_price', 1]
        
        # Get bucket
        bucket = cat_data['bucket'].iloc[0]
        
        # Store results
        category_results.append({
            'Category': category,
            'Bucket': bucket,
            'Elasticity': elasticity,
            'Std_Error': std_err,
            't_statistic': t_stat,
            'p_value': p_value,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper,
            'Significant': p_value < 0.05,
            'R_squared': model.rsquared,
            'Observations': model.nobs
        })
        
    except Exception as e:
        print(f"  ⚠️  {category}: Error - {str(e)[:50]}")
        continue

# ----------------------------------------
# Create results dataframe
# ----------------------------------------
elasticity_df = pd.DataFrame(category_results)

if len(elasticity_df) > 0:
    # Sort by elasticity (most elastic first)
    elasticity_df = elasticity_df.sort_values('Elasticity')
    
    print(f"\n📊 Successfully estimated {len(elasticity_df)} category elasticities")
    print("\n" + "-"*100)
    print(f"{'Category':<30} {'Bucket':<20} {'Elasticity':>10} {'Std Err':>10} {'p-value':>10} {'Sig':>5}")
    print("-"*100)
    
    for _, row in elasticity_df.iterrows():
        sig_marker = '***' if row['p_value'] < 0.01 else '**' if row['p_value'] < 0.05 else '*' if row['p_value'] < 0.10 else ''
        print(f"{row['Category']:<30} {row['Bucket']:<20} {row['Elasticity']:>10.3f} {row['Std_Error']:>10.3f} {row['p_value']:>10.4f} {sig_marker:>5}")
    
    # ----------------------------------------
    # Summary statistics
    # ----------------------------------------
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    
    print(f"\nElasticity Distribution:")
    print(f"  • Mean: {elasticity_df['Elasticity'].mean():.3f}")
    print(f"  • Median: {elasticity_df['Elasticity'].median():.3f}")
    print(f"  • Std Dev: {elasticity_df['Elasticity'].std():.3f}")
    print(f"  • Min: {elasticity_df['Elasticity'].min():.3f} ({elasticity_df.loc[elasticity_df['Elasticity'].idxmin(), 'Category']})")
    print(f"  • Max: {elasticity_df['Elasticity'].max():.3f} ({elasticity_df.loc[elasticity_df['Elasticity'].idxmax(), 'Category']})")
    
    print(f"\nStatistical Significance:")
    print(f"  • Significant at 1%: {(elasticity_df['p_value'] < 0.01).sum()}")
    print(f"  • Significant at 5%: {(elasticity_df['p_value'] < 0.05).sum()}")
    print(f"  • Significant at 10%: {(elasticity_df['p_value'] < 0.10).sum()}")
    print(f"  • Not significant: {(elasticity_df['p_value'] >= 0.10).sum()}")
    
    # ----------------------------------------
    # By bucket
    # ----------------------------------------
    print("\n📊 Elasticity by Bucket:")
    bucket_stats = elasticity_df.groupby('Bucket').agg({
        'Elasticity': ['count', 'mean', 'std', 'min', 'max']
    }).round(3)
    print(bucket_stats)
    
    # ----------------------------------------
    # Save results
    # ----------------------------------------
    elasticity_df.to_csv('../outputs/category_elasticities.csv', index=False)
    print("\n✅ Results saved to: ../outputs/category_elasticities.csv")
    
else:
    print("\n⚠️  No categories had sufficient data for estimation")

print("\n✅ CATEGORY-LEVEL ELASTICITY ESTIMATION COMPLETE!")

In [ ]:
# ============================================
# VISUALIZE CATEGORY-LEVEL ELASTICITIES
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns

if len(elasticity_df) > 0:
    print("\n📊 Creating category elasticity visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Category-Level Price Elasticities', fontsize=16, fontweight='bold')
    
    # ----------------------------------------
    # Plot 1: Elasticity Point Estimates
    # ----------------------------------------
    ax1 = axes[0, 0]
    
    # Sort and plot
    plot_data = elasticity_df.sort_values('Elasticity').head(15)  # Top 15 most elastic
    
    y_pos = range(len(plot_data))
    elasticities = plot_data['Elasticity'].values
    ci_lower = plot_data['CI_Lower'].values
    ci_upper = plot_data['CI_Upper'].values
    errors = np.array([elasticities - ci_lower, ci_upper - elasticities])
    
    # Color by bucket
    bucket_colors = {'LEISURE_LIFESTYLE': '#e74c3c', 'HOME_ESSENTIALS': '#3498db', 'PERSONAL_CARE': '#2ecc71'}
    colors = [bucket_colors.get(b, '#95a5a6') for b in plot_data['Bucket']]
    
    ax1.barh(y_pos, elasticities, color=colors, alpha=0.7, edgecolor='black')
    ax1.errorbar(elasticities, y_pos, xerr=errors, fmt='none', ecolor='black', capsize=5, capthick=2)
    
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(plot_data['Category'], fontsize=9)
    ax1.set_xlabel('Price Elasticity', fontsize=12, fontweight='bold')
    ax1.set_title('Top 15 Categories by Elasticity (with 95% CI)', fontsize=12, fontweight='bold')
    ax1.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax1.axvline(x=-1, color='gray', linestyle='--', linewidth=2, alpha=0.5, label='Unit Elastic')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='x')
    
    # ----------------------------------------
    # Plot 2: Elasticity Distribution
    # ----------------------------------------
    ax2 = axes[0, 1]
    
    ax2.hist(elasticity_df['Elasticity'], bins=15, edgecolor='black', alpha=0.7, color='steelblue')
    ax2.axvline(elasticity_df['Elasticity'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {elasticity_df["Elasticity"].mean():.2f}')
    ax2.axvline(elasticity_df['Elasticity'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {elasticity_df["Elasticity"].median():.2f}')
    ax2.set_xlabel('Elasticity', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title('Distribution of Category Elasticities', fontsize=12, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # ----------------------------------------
    # Plot 3: Elasticity by Bucket
    # ----------------------------------------
    ax3 = axes[1, 0]
    
    bucket_means = elasticity_df.groupby('Bucket')['Elasticity'].mean().sort_values()
    bucket_colors_list = [bucket_colors.get(b, '#95a5a6') for b in bucket_means.index]
    
    ax3.barh(range(len(bucket_means)), bucket_means.values, color=bucket_colors_list, alpha=0.7, edgecolor='black')
    ax3.set_yticks(range(len(bucket_means)))
    ax3.set_yticklabels(bucket_means.index)
    ax3.set_xlabel('Mean Elasticity', fontsize=12, fontweight='bold')
    ax3.set_title('Average Elasticity by Bucket', fontsize=12, fontweight='bold')
    ax3.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax3.axvline(x=-1, color='gray', linestyle='--', linewidth=2, alpha=0.5)
    ax3.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, v in enumerate(bucket_means.values):
        ax3.text(v - 0.1, i, f'{v:.2f}', va='center', ha='right', fontweight='bold', fontsize=10)
    
    # ----------------------------------------
    # Plot 4: Significance vs Elasticity
    # ----------------------------------------
    ax4 = axes[1, 1]
    
    # Color by significance
    sig_colors = ['green' if p < 0.05 else 'orange' if p < 0.10 else 'red' for p in elasticity_df['p_value']]
    
    ax4.scatter(elasticity_df['Elasticity'], elasticity_df['t_statistic'], 
                c=sig_colors, alpha=0.6, s=100, edgecolor='black')
    ax4.axhline(y=1.96, color='green', linestyle='--', linewidth=2, alpha=0.5, label='p=0.05 threshold')
    ax4.axhline(y=-1.96, color='green', linestyle='--', linewidth=2, alpha=0.5)
    ax4.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax4.set_xlabel('Elasticity', fontsize=12, fontweight='bold')
    ax4.set_ylabel('t-statistic', fontsize=12, fontweight='bold')
    ax4.set_title('Statistical Significance Check', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Add legend for colors
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='green', label='p < 0.05 (Significant)'),
        Patch(facecolor='orange', label='0.05 < p < 0.10'),
        Patch(facecolor='red', label='p > 0.10 (Not Significant)')
    ]
    ax4.legend(handles=legend_elements, loc='best', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../outputs/category_elasticities.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Visualizations complete and saved!")
    print("   Saved to: ../outputs/category_elasticities.png")

else:
    print("⚠️  No elasticity results to visualize")

In [ ]:
# ============================================
# NESTED LOGIT DATA PREPARATION
# ============================================

print("\n" + "="*60)
print("NESTED LOGIT: DATA PREPARATION")
print("="*60)

# ----------------------------------------
# Concept: Each purchase = choice among buckets
# Then within bucket, choice among products
# ----------------------------------------

print("\n📊 Creating choice set structure...")

# Start with orders_full_extended (individual purchases)
nested_data = orders_full_extended[
    ['order_id', 'customer_unique_id', 'product_id', 'bucket', 
     'price', 'freight_value', 'review_score', 
     'product_weight_g', 'customer_state', 'seller_state',
     'order_purchase_timestamp']
].copy()

# Remove missing values
nested_data = nested_data.dropna(subset=['bucket', 'price'])

print(f"  • Total purchases: {len(nested_data):,}")
print(f"  • Unique customers: {nested_data['customer_unique_id'].nunique():,}")
print(f"  • Unique products: {nested_data['product_id'].nunique():,}")
print(f"  • Buckets: {nested_data['bucket'].nunique()}")

# ----------------------------------------
# Calculate bucket-level statistics
# ----------------------------------------
print("\n📈 Calculating bucket-level attributes...")

# Average price per bucket
bucket_avg_price = nested_data.groupby('bucket')['price'].mean().to_dict()
nested_data['bucket_avg_price'] = nested_data['bucket'].map(bucket_avg_price)

# Number of products per bucket
bucket_num_products = nested_data.groupby('bucket')['product_id'].nunique().to_dict()
nested_data['bucket_num_products'] = nested_data['bucket'].map(bucket_num_products)

# Average review score per bucket
bucket_avg_review = nested_data.groupby('bucket')['review_score'].mean().to_dict()
nested_data['bucket_avg_review'] = nested_data['bucket'].map(bucket_avg_review)

print("  ✓ Bucket-level variables created")

# ----------------------------------------
# Sample for computational efficiency
# ----------------------------------------
# Nested logit on full data is computationally intensive
# Sample 20,000 purchases for estimation

np.random.seed(42)
sample_size = min(20000, len(nested_data))
nested_sample = nested_data.sample(n=sample_size, random_state=42)

print(f"\n📋 Sampled {sample_size:,} purchases for estimation")
print(f"  • Buckets in sample: {nested_sample['bucket'].nunique()}")
print(f"  • Customers in sample: {nested_sample['customer_unique_id'].nunique():,}")

# ----------------------------------------
# Create market shares (Stage 1: Bucket choice)
# ----------------------------------------
print("\n📊 Calculating market shares...")

# Total purchases per time period (month)
nested_sample['year_month'] = pd.to_datetime(nested_sample['order_purchase_timestamp']).dt.to_period('M')

# Bucket shares by month
bucket_shares = nested_sample.groupby(['year_month', 'bucket']).size().reset_index(name='bucket_count')
total_monthly = nested_sample.groupby('year_month').size().reset_index(name='total_count')

bucket_shares = bucket_shares.merge(total_monthly, on='year_month')
bucket_shares['bucket_share'] = bucket_shares['bucket_count'] / bucket_shares['total_count']

print(f"  • Market share observations: {len(bucket_shares)}")

# ----------------------------------------
# Save prepared data
# ----------------------------------------
nested_sample.to_pickle('../data/processed/nested_logit_sample.pkl')
bucket_shares.to_pickle('../data/processed/bucket_shares.pkl')

print("\n✅ Nested logit data prepared and saved!")

In [ ]:
# ============================================
# NESTED LOGIT: STAGE 1 - BUCKET CHOICE
# ============================================

import statsmodels.api as sm
import statsmodels.formula.api as smf

print("\n" + "="*60)
print("STAGE 1: BUCKET CHOICE MODEL")
print("="*60)

# ----------------------------------------
# Prepare bucket-level choice data
# ----------------------------------------
print("\n📊 Preparing bucket choice data...")

# Calculate inclusive value (from Stage 2 - simplified for now)
# In full nested logit, this comes from within-bucket choices
# For simplified version, use bucket average attributes

bucket_choice_data = bucket_shares.merge(
    nested_sample.groupby(['year_month', 'bucket']).agg({
        'price': 'mean',
        'review_score': 'mean',
        'product_id': 'nunique',
        'bucket_num_products': 'first'
    }).reset_index(),
    on=['year_month', 'bucket']
)

# Log transformations
bucket_choice_data['log_price'] = np.log(bucket_choice_data['price'])
bucket_choice_data['log_products'] = np.log(bucket_choice_data['product_id'])
bucket_choice_data['log_share'] = np.log(bucket_choice_data['bucket_share'])

print(f"  • Observations: {len(bucket_choice_data)}")
print(f"  • Buckets: {bucket_choice_data['bucket'].nunique()}")
print(f"  • Time periods: {bucket_choice_data['year_month'].nunique()}")

# ----------------------------------------
# Estimate bucket choice model
# ----------------------------------------
print("\n📈 Estimating bucket choice model...")

# Simple logit specification (market share model)
# log(share) = β0 + β1*log(price) + β2*log(products) + controls

bucket_model = smf.ols(
    'log_share ~ log_price + log_products + review_score + C(year_month)',
    data=bucket_choice_data
).fit()

print("\n" + "-"*60)
print("BUCKET CHOICE RESULTS")
print("-"*60)

print(f"\nPrice coefficient (bucket-level elasticity):")
print(f"  • Coefficient: {bucket_model.params['log_price']:.3f}")
print(f"  • Std Error: {bucket_model.bse['log_price']:.3f}")
print(f"  • t-statistic: {bucket_model.tvalues['log_price']:.3f}")
print(f"  • p-value: {bucket_model.pvalues['log_price']:.4f}")

print(f"\nProduct variety coefficient:")
print(f"  • Coefficient: {bucket_model.params['log_products']:.3f}")
print(f"  • Std Error: {bucket_model.bse['log_products']:.3f}")

print(f"\nModel fit:")
print(f"  • R-squared: {bucket_model.rsquared:.3f}")
print(f"  • Observations: {bucket_model.nobs:.0f}")

# ----------------------------------------
# Bucket-specific shares
# ----------------------------------------
print("\n📊 Average bucket shares:")
avg_shares = bucket_shares.groupby('bucket')['bucket_share'].mean().sort_values(ascending=False)
print(avg_shares.round(3))

print("\n✅ Stage 1 (Bucket Choice) complete!")

In [ ]:
# ============================================
# CROSS-BUCKET SUBSTITUTION PATTERNS
# ============================================

print("\n" + "="*60)
print("CROSS-BUCKET SUBSTITUTION ANALYSIS")
print("="*60)

# ----------------------------------------
# Analyze repeat customers who switched buckets
# ----------------------------------------
print("\n📊 Analyzing bucket switching behavior...")

# Get repeat customers
repeat_customers = nested_sample.groupby('customer_unique_id').filter(lambda x: len(x) >= 2)

print(f"  • Repeat purchases in sample: {len(repeat_customers):,}")
print(f"  • Repeat customers: {repeat_customers['customer_unique_id'].nunique():,}")

# ----------------------------------------
# Calculate bucket transitions
# ----------------------------------------
# For each customer, look at bucket sequence

transitions = []

for customer_id in repeat_customers['customer_unique_id'].unique():
    customer_data = repeat_customers[repeat_customers['customer_unique_id'] == customer_id].sort_values('order_purchase_timestamp')
    
    buckets = customer_data['bucket'].tolist()
    
    # Create transitions (from bucket i to bucket j)
    for i in range(len(buckets) - 1):
        transitions.append({
            'from_bucket': buckets[i],
            'to_bucket': buckets[i+1],
            'same_bucket': buckets[i] == buckets[i+1]
        })

transition_df = pd.DataFrame(transitions)

print(f"\n📈 Transition patterns:")
print(f"  • Total transitions: {len(transition_df):,}")
print(f"  • Same bucket: {transition_df['same_bucket'].sum():,} ({transition_df['same_bucket'].mean()*100:.1f}%)")
print(f"  • Different bucket: {(~transition_df['same_bucket']).sum():,} ({(~transition_df['same_bucket']).mean()*100:.1f}%)")

# ----------------------------------------
# Create transition matrix
# ----------------------------------------
print("\n📊 Bucket transition matrix:")

# Count transitions
transition_matrix = pd.crosstab(
    transition_df['from_bucket'], 
    transition_df['to_bucket'],
    normalize='index'  # Row percentages
)

print("\nProbability of switching FROM bucket (row) TO bucket (column):")
print(transition_matrix.round(3))

# ----------------------------------------
# Visualize transition matrix
# ----------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(
    transition_matrix, 
    annot=True, 
    fmt='.3f', 
    cmap='YlOrRd',
    cbar_kws={'label': 'Transition Probability'},
    linewidths=0.5,
    ax=ax
)

ax.set_title('Bucket-to-Bucket Transition Matrix\n(Probability customer switches from row bucket to column bucket)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('To Bucket (Next Purchase)', fontsize=12, fontweight='bold')
ax.set_ylabel('From Bucket (Previous Purchase)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/bucket_transition_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Transition matrix saved to: ../outputs/bucket_transition_matrix.png")

# ----------------------------------------
# Key substitution patterns
# ----------------------------------------
print("\n" + "="*60)
print("KEY SUBSTITUTION INSIGHTS")
print("="*60)

# Most common cross-bucket switches
cross_bucket = transition_df[~transition_df['same_bucket']]
top_switches = cross_bucket.groupby(['from_bucket', 'to_bucket']).size().sort_values(ascending=False).head(10)

print("\nTop 10 cross-bucket switches:")
for (from_b, to_b), count in top_switches.items():
    pct = count / len(cross_bucket) * 100
    print(f"  {from_b} → {to_b}: {count} switches ({pct:.1f}%)")

print("\n✅ CROSS-BUCKET SUBSTITUTION ANALYSIS COMPLETE!")

In [ ]:
# ============================================
# CROSS-BUCKET ELASTICITIES
# ============================================

print("\n" + "="*60)
print("CROSS-BUCKET PRICE ELASTICITIES")
print("="*60)

# ----------------------------------------
# Estimate how bucket i's share responds to bucket j's price
# ----------------------------------------

print("\n📊 Estimating cross-bucket elasticities...")

# Focus on top 4 buckets for tractability
top_buckets = ['LEISURE_LIFESTYLE', 'HOME_ESSENTIALS', 'PERSONAL_CARE', 'ELECTRONICS_TECH']

# Create wide format with bucket shares and prices
bucket_wide = bucket_choice_data.pivot_table(
    index='year_month',
    columns='bucket',
    values=['bucket_share', 'price']
).reset_index()

# Flatten column names
bucket_wide.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in bucket_wide.columns.values]

print(f"  • Observations: {len(bucket_wide)}")
print(f"  • Variables: {len(bucket_wide.columns)}")

# ----------------------------------------
# Estimate cross-elasticities
# ----------------------------------------
# For each bucket, estimate how its share responds to other buckets' prices

cross_elasticities = []

for target_bucket in top_buckets:
    if f'bucket_share_{target_bucket}' not in bucket_wide.columns:
        continue
    
    # Dependent variable: log share of target bucket
    y_col = f'bucket_share_{target_bucket}'
    if bucket_wide[y_col].min() <= 0:
        continue  # Skip if any zero shares
    
    bucket_wide[f'log_share_{target_bucket}'] = np.log(bucket_wide[y_col])
    
    # Independent variables: log prices of all buckets
    price_cols = []
    for b in top_buckets:
        price_col = f'price_{b}'
        if price_col in bucket_wide.columns:
            bucket_wide[f'log_price_{b}'] = np.log(bucket_wide[price_col])
            price_cols.append(f'log_price_{b}')
    
    if len(price_cols) < 2:
        continue
    
    # Regression
    formula = f'log_share_{target_bucket} ~ {" + ".join(price_cols)}'
    
    try:
        model = smf.ols(formula, data=bucket_wide).fit()
        
        # Extract elasticities
        for price_var in price_cols:
            source_bucket = price_var.replace('log_price_', '')
            
            cross_elasticities.append({
                'Target_Bucket': target_bucket,
                'Source_Bucket': source_bucket,
                'Elasticity': model.params[price_var],
                'Std_Error': model.bse[price_var],
                'p_value': model.pvalues[price_var],
                'Type': 'Own' if source_bucket == target_bucket else 'Cross'
            })
    except:
        continue

# ----------------------------------------
# Create elasticity matrix
# ----------------------------------------
if len(cross_elasticities) > 0:
    elasticity_df = pd.DataFrame(cross_elasticities)
    
    # Pivot to matrix format
    elasticity_matrix = elasticity_df.pivot(
        index='Target_Bucket',
        columns='Source_Bucket',
        values='Elasticity'
    )
    
    print("\n" + "="*60)
    print("CROSS-BUCKET ELASTICITY MATRIX")
    print("="*60)
    print("\nHow does row bucket's demand respond to column bucket's price?")
    print("(Diagonal = own-price elasticity, Off-diagonal = cross-price)")
    print()
    print(elasticity_matrix.round(3))
    
    # Highlight substitutes vs complements
    print("\n📊 Interpretation:")
    print("  • Negative own-price elasticity (diagonal) = normal demand response")
    print("  • Positive cross-price elasticity = substitutes")
    print("  • Negative cross-price elasticity = complements")
    
    # Save
    elasticity_df.to_csv('../outputs/cross_bucket_elasticities.csv', index=False)
    print("\n✅ Results saved to: ../outputs/cross_bucket_elasticities.csv")
    
    # ----------------------------------------
    # Visualize
    # ----------------------------------------
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(
        elasticity_matrix,
        annot=True,
        fmt='.2f',
        cmap='RdBu_r',
        center=0,
        cbar_kws={'label': 'Elasticity'},
        linewidths=0.5,
        ax=ax
    )
    
    ax.set_title('Cross-Bucket Price Elasticity Matrix\n(Row bucket demand response to column bucket price)', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Price Change Bucket', fontsize=12, fontweight='bold')
    ax.set_ylabel('Demand Response Bucket', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../outputs/cross_bucket_elasticity_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Visualization saved!")

else:
    print("\n⚠️  Insufficient data for cross-elasticity estimation")

print("\n✅ NESTED LOGIT ANALYSIS COMPLETE!")